# XGBoost Model Training Grid Search CV - Elemental Features




## Import Libraries and Setup

In [ ]:
# Import required libraries
import pandas as pd
import numpy as np
from sklearn.model_selection import KFold, GridSearchCV, RandomizedSearchCV
import xgboost as xgb
import matplotlib.pyplot as plt
from sklearn.metrics import mean_absolute_error, mean_absolute_percentage_error, r2_score, mean_squared_error
from sklearn.inspection import permutation_importance
import pickle
import os
import warnings
import time
from concurrent.futures import ThreadPoolExecutor, ProcessPoolExecutor
import multiprocessing as mp
warnings.filterwarnings('ignore')

# Set random state
SEP = os.sep
random_state = 21

print("Libraries imported successfully!")

## Load and Prepare Data

In [ ]:
# Load data
print("Loading data...")
df = pd.read_excel("PGM_CMP_01_norm_feature.xlsx").sample(frac=1, random_state=random_state, ignore_index=True)


print("\nFirst 5 rows of data:")
display(df.head())

# Define features and target
y_prop = "Mass_Change"
feat_names = ['Time', 'Ni', 'Al', 'Pt', 'Ir', 'Pd', 'Rh']

X = np.array(df[feat_names])
Y = np.array(df[y_prop])

print(f"\nX shape: {X.shape}")
print(f"Y shape: {Y.shape}")
print(f"\nFeatures: {feat_names}")
print(f"Target: {y_prop}")


## Create Output Directories

In [ ]:
# Create directories for saving results
directories = [
    "trained_mod_CBFV_xgboost",
    "xgboost_parity_plots",
    "xgboost_results"
]

for directory in directories:
    os.makedirs(directory, exist_ok=True)
    print(f"✓ Created directory: {directory}")

print("\nAll directories ready!")

## Define Extended Hyperparameter Grid (Optimized for GPU)

In [ ]:


param_grid = {
    'n_estimators': [1000, 2000, 2500], 
    'max_depth': [6, 8, 10, 12], 
    'learning_rate': [0.001, 0.01, 0.05],
    'min_child_weight': [1, 2, 3, 5],
    'subsample': [0.5, 0.65, 0.95, 0.85,  0.75],
    'colsample_bytree': [0.5, 0.65, 0.95, 0.85,  0.75],
    'gamma': [0, 0.05, 0.1],
    'reg_alpha': [0, 0.01, 0.1, 0.5, 1.0], 
    'reg_lambda': [0.5, 1.0, 2.0, 5.0, 1.5], 
}

print("=" * 70)
print("HYPERPARAMETER GRID (OPTIMIZED FOR GPU)")
print("=" * 70)
print("\nParameter Grid:")
for param, values in param_grid.items():
    print(f"  {param}: {values}")

total_combinations = np.prod([len(v) for v in param_grid.values()])
print(f"\nTotal parameter combinations: {total_combinations:,}")
print("\nNote: Using RandomizedSearchCV for efficient sampling")
print("Higher n_estimators and max_depth values will maximize GPU utilization")

## Run GRIDSearchCV for Faster Hyperparameter Tuning

In [ ]:
print('=' * 70)
print('STARTING GRID SEARCH (CPU-ACCELERATED)')
print('=' * 70)

from sklearn.model_selection import ParameterGrid
import time
import multiprocessing as mp
import xgboost as xgb

# Total fits = combinations × CV folds (5)
total_fits     = len(list(ParameterGrid(param_grid))) * 5
total_combinations = len(list(ParameterGrid(param_grid)))
fits_done      = [0]
eta_start_time = [None]
UPDATE_EVERY   = 100  


class ETACallback:
    """
    CPU version: Counts GridSearchCV fits and prints live ETA.
    Works with n_jobs=-1 (parallel CPU) but ETA is approximate.
    """
    def __init__(self, estimator):
        self.estimator = estimator
        for attr in ('get_params', 'set_params', '_get_tags',
                     '__sklearn_tags__', '_more_tags'):
            if hasattr(estimator, attr):
                setattr(self, attr, getattr(estimator, attr))

    def fit(self, X, y, **kw):
        if eta_start_time[0] is None:
            eta_start_time[0] = time.time()

        self.estimator.fit(X, y, **kw)
        fits_done[0] += 1

        if fits_done[0] % UPDATE_EVERY == 0 or fits_done[0] == total_fits:
            elapsed   = time.time() - eta_start_time[0]
            avg_t     = elapsed / max(fits_done[0], 1)
            remaining = total_fits - fits_done[0]
            eta_sec   = avg_t * remaining
            pct       = 100.0 * fits_done[0] / total_fits

            eta_str = (
                f'{eta_sec/3600:.1f}h' if eta_sec >= 3600
                else f'{eta_sec/60:.1f}m' if eta_sec >= 60
                else f'{eta_sec:.0f}s'
            )

            print(
                f'\r  [{fits_done[0]:>6}/{total_fits}]  {pct:5.1f}%  '
                f'Elapsed: {elapsed/60:6.1f}m  ETA: {eta_str}   ',
                end='', flush=True
            )
        return self

    def predict(self, X): 
        return self.estimator.predict(X)

    def score(self, X, y): 
        return self.estimator.score(X, y)

    def __getattr__(self, name): 
        return getattr(self.estimator, name)



xgb_base = xgb.XGBRegressor(
    random_state=random_state,
    objective='reg:squarederror',
    tree_method='hist',   
    n_jobs=mp.cpu_count() 
)

# GridSearchCV 
grid_search = GridSearchCV(
    estimator=ETACallback(xgb_base),
    param_grid=param_grid,
    scoring='neg_mean_absolute_error',
    cv=5,
    verbose=0,
    n_jobs=50,  
    return_train_score=True,
)

print(f'\nDevice              : CPU')
print(f'CPU cores used      : {mp.cpu_count()}')
print(f'Total fits to run   : {total_fits:,}  ({total_combinations:,} combos × 5 folds)')
print(f'ETA updates every   : {UPDATE_EVERY} fits')
print('\nProgress:')

start_time = time.time()
grid_search.fit(X, Y)
grid_search_time = time.time() - start_time
search_time = grid_search_time

print()  
print('\n' + '=' * 70)
print('GRID SEARCH COMPLETED')
print('=' * 70)
print(f'\nTotal time : {grid_search_time/60:.2f} minutes')
print(f'Best CV MAE: {-grid_search.best_score_:.6f}')

print('\nBest Parameters Found:')
for param, value in grid_search.best_params_.items():
    print(f'  {param}: {value}')

## Extract and Save Top Parameter Sets

In [ ]:
# Extract top parameter combinations
cv_results = pd.DataFrame(grid_search.cv_results_)
cv_results['mean_test_mae'] = -cv_results['mean_test_score']
cv_results_sorted = cv_results.sort_values('mean_test_mae').head(10)

# Display top 10 parameter sets
print("=" * 70)
print("TOP 10 PARAMETER COMBINATIONS")
print("=" * 70)

top_params_df = cv_results_sorted[['params', 'mean_test_mae', 'std_test_score']].reset_index(drop=True)
top_params_df.index = range(1, 11)
display(top_params_df)

# Save to CSV
top_params_df.to_csv('xgboost_results/top_10_parameter_sets.csv')
print("\n✓ Saved to: xgboost_results/top_10_parameter_sets.csv")

## K-Fold Cross-Validation with GPU Optimization

In [ ]:
print("=" * 70)
print("TESTING TOP PARAMETER SETS WITH K-FOLD CV (CPU-ONLY)")
print("=" * 70)

# Select top 5 parameter sets
top_n_sets = 5
top_param_sets = cv_results_sorted.head(top_n_sets)['params'].tolist()
all_param_set_results = []

# K-Fold configuration
Kfold_val = 5

# Function to train a single fold
def train_fold(fold_data):
    """Train a single fold - CPU version"""
    x_train, x_test, y_train, y_test, param_set, fold_idx, set_idx = fold_data
    
    # Create model 
    xgb_params = {
        **param_set,
        'random_state': random_state,
        'objective': 'reg:squarederror',
        'tree_method': 'hist',  
        'n_jobs': -1           
    }
    
    # Train using sklearn API 
    xgb_mod = xgb.XGBRegressor(**xgb_params)
    xgb_mod.fit(
        x_train, y_train,
        eval_set=[(x_train, y_train), (x_test, y_test)],
        verbose=False
    )
    
    # Predictions
    y_train_pred = xgb_mod.predict(x_train)
    y_test_pred = xgb_mod.predict(x_test)
    
    return (y_train_pred, y_test_pred, xgb_mod)

for set_idx, param_set in enumerate(top_param_sets, 1):
    print(f"\n{'='*70}")
    print(f"PARAMETER SET {set_idx}/{top_n_sets}")
    print(f"{'='*70}")
    print("Parameters:")
    for param, value in param_set.items():
        print(f"  {param}: {value}")
    
    # Initialize K-Fold
    kf = KFold(n_splits=Kfold_val, shuffle=True, random_state=random_state)
    
    # Initialize performance metrics storage
    metrics = {
        'train_mae': [], 'test_mae': [],
        'train_mse': [], 'test_mse': [],
        'train_rmse': [], 'test_rmse': [],
        'train_mape': [], 'test_mape': [],
        'train_r2': [], 'test_r2': []
    }
    
    k_count = 1
    all_predictions = {'train': [], 'test': [], 'y_train': [], 'y_test': []}
    
  
    set_start_time = time.time()
    
    for train_indices, test_indices in kf.split(X):
        print(f"\n  Fold {k_count}/{Kfold_val}...", end="")
        
        # Split data
        x_train, x_test = X[train_indices], X[test_indices]
        y_train, y_test = Y[train_indices], Y[test_indices]
        
        # Train fold 
        fold_start = time.time()
        y_train_pred, y_test_pred, xgb_mod = train_fold(
            (x_train, x_test, y_train, y_test, param_set, k_count, set_idx)
        )
        fold_time = time.time() - fold_start
        
        # Store predictions
        all_predictions['train'].extend(y_train_pred)
        all_predictions['test'].extend(y_test_pred)
        all_predictions['y_train'].extend(y_train)
        all_predictions['y_test'].extend(y_test)
        
        # Calculate metrics
        metrics['train_mae'].append(round(mean_absolute_error(y_train, y_train_pred), 6))
        metrics['test_mae'].append(round(mean_absolute_error(y_test, y_test_pred), 6))
        
        train_mse = mean_squared_error(y_train, y_train_pred)
        test_mse = mean_squared_error(y_test, y_test_pred)
        metrics['train_mse'].append(round(train_mse, 6))
        metrics['test_mse'].append(round(test_mse, 6))
        
        metrics['train_rmse'].append(round(np.sqrt(train_mse), 6))
        metrics['test_rmse'].append(round(np.sqrt(test_mse), 6))
        
        metrics['train_mape'].append(round(mean_absolute_percentage_error(y_train, y_train_pred), 4))
        metrics['test_mape'].append(round(mean_absolute_percentage_error(y_test, y_test_pred), 4))
        
        train_r2 = r2_score(y_train, y_train_pred)
        test_r2 = r2_score(y_test, y_test_pred)
        metrics['train_r2'].append(round(train_r2, 4))
        metrics['test_r2'].append(round(test_r2, 4))
        
        # Save model
        model_savename = f"trained_mod_CBFV_xgboost{SEP}xgb-mod-set{set_idx:02d}-K{k_count}.pkl"
        pickle.dump(xgb_mod, open(model_savename, 'wb'))
        
        print(f" MAE: {metrics['test_mae'][-1]:.6f}, R²: {test_r2:.4f} ({fold_time:.2f}s)")
        k_count += 1
    
    set_time = time.time() - set_start_time
    print(f"\n  Parameter set {set_idx} completed in: {set_time:.2f} seconds")
    
    # Calculate summary statistics
    print(f"\n  Performance Summary:")
    print("  " + "-"*66)
    
    evaluation_metrics = ['mae', 'mse', 'rmse', 'mape', 'r2']
    set_summary = {'param_set': set_idx}
    
    for metric in evaluation_metrics:
        train_key = f'train_{metric}'
        test_key = f'test_{metric}'
        
        train_mean = np.mean(metrics[train_key])
        train_std = np.std(metrics[train_key])
        test_mean = np.mean(metrics[test_key])
        test_std = np.std(metrics[test_key])
        
        set_summary[f'train_{metric}_mean'] = train_mean
        set_summary[f'train_{metric}_std'] = train_std
        set_summary[f'test_{metric}_mean'] = test_mean
        set_summary[f'test_{metric}_std'] = test_std
        
        print(f"  {metric.upper()}:")
        print(f"    Train: {train_mean:.6f} ± {train_std:.6f}")
        print(f"    Test:  {test_mean:.6f} ± {test_std:.6f}")
    
    all_param_set_results.append(set_summary)
    
    # Save detailed results
    results_df = pd.DataFrame(metrics)
    results_df.index = [f'Fold_{i+1}' for i in range(Kfold_val)]
    results_df.to_csv(f'xgboost_results/cv_results_set{set_idx:02d}.csv')
    
    # Save feature importance 
    try:
        importance_scores = xgb_mod.feature_importances_
        
        feature_importance_df = pd.DataFrame({
            'Feature': feat_names,
            'Importance': importance_scores
        }).sort_values('Importance', ascending=False)
        
        feature_importance_df.to_csv(
            f'xgboost_results/feature_importance_set{set_idx:02d}.csv',
            index=False
        )
    except Exception as e:
        print(f"\n  Warning: Could not extract feature importance: {e}")
    
    
    locals()[f'metrics_set{set_idx}'] = metrics
    locals()[f'predictions_set{set_idx}'] = all_predictions
    locals()[f'params_set{set_idx}'] = param_set


print("\n" + "=" * 70)
print("K-FOLD CROSS-VALIDATION COMPLETED (CPU)")
print("=" * 70)

## Generate Parity Plots for Each Parameter Set

In [ ]:
print("=" * 70)
print("GENERATING PARITY PLOTS (CPU)")
print("=" * 70)

for set_idx in range(1, top_n_sets + 1):
    # Retrieve stored data
    metrics = locals()[f'metrics_set{set_idx}']
    all_predictions = locals()[f'predictions_set{set_idx}']
    param_set = locals()[f'params_set{set_idx}']
    
    # Create parity plot
    fig, ax = plt.subplots(figsize=(10, 10))
    
    # Calculate statistics
    train_r2_mean = np.mean(metrics['train_r2'])
    test_r2_mean = np.mean(metrics['test_r2'])
    train_mae_mean = np.mean(metrics['train_mae'])
    test_mae_mean = np.mean(metrics['test_mae'])
    
    # Scatter plots
    ax.scatter(
        all_predictions['y_train'], all_predictions['train'],
        alpha=0.6, s=60,
        label=f"Train (R²={train_r2_mean:.3f}, MAE={train_mae_mean:.4f})",
        edgecolors='black', linewidths=0.5
    )
    
    ax.scatter(
        all_predictions['y_test'], all_predictions['test'],
        alpha=0.6, s=60,
        label=f"Test (R²={test_r2_mean:.3f}, MAE={test_mae_mean:.4f})",
        edgecolors='black', linewidths=0.5
    )
    
    # Perfect prediction line
    y_min, y_max = min(Y), max(Y)
    ax.plot(
        [y_min, y_max], [y_min, y_max],
        'k--', lw=2.5,
        label='Perfect Prediction', alpha=0.7
    )
    
    # Formatting
    ax.set_xlabel("Experimental Mass Change", fontsize=14, fontweight='bold')
    ax.set_ylabel("Predicted Mass Change", fontsize=14, fontweight='bold')
    
    ax.set_title(
        f"Parity Plot - Parameter Set {set_idx} [CPU]\n"
        f"n_est={param_set['n_estimators']}, "
        f"max_d={param_set['max_depth']}, "
        f"lr={param_set['learning_rate']}",
        fontsize=14,
        fontweight='bold'
    )
    
    ax.legend(fontsize=11, loc='upper left', framealpha=0.9)
    ax.grid(True, alpha=0.3, linestyle='--')
    ax.set_aspect('equal', adjustable='box')
    
    plt.tight_layout()
    plt.savefig(
        f'xgboost_parity_plots/parity_plot_set{set_idx:02d}.png',
        dpi=300,
        bbox_inches='tight'
    )
    plt.show()
    
    print(f"✓ Saved: xgboost_parity_plots/parity_plot_set{set_idx:02d}.png")

print("\nAll parity plots generated (CPU)!")

## Generate Detailed Metrics Plots

In [ ]:
print("=" * 70)
print("GENERATING DETAILED METRICS PLOTS")
print("=" * 70)

for set_idx in range(1, top_n_sets + 1):
    metrics = locals()[f'metrics_set{set_idx}']
    
    fig, axes = plt.subplots(2, 3, figsize=(16, 10))
    axes = axes.flatten()
    
    metric_titles = ['MAE', 'MSE', 'RMSE', 'MAPE', 'R² Score']
    evaluation_metrics = ['mae', 'mse', 'rmse', 'mape', 'r2']
    colors_train = plt.cm.Blues(0.7)
    colors_test = plt.cm.Oranges(0.7)
    
    for idx, metric in enumerate(evaluation_metrics):
        train_key = f'train_{metric}'
        test_key = f'test_{metric}'
        
        x_pos = np.arange(Kfold_val)
        width = 0.35
        
        bars1 = axes[idx].bar(x_pos - width/2, metrics[train_key], width, 
                              label='Train', alpha=0.8, color=colors_train, 
                              edgecolor='black', linewidth=1.2)
        bars2 = axes[idx].bar(x_pos + width/2, metrics[test_key], width, 
                              label='Test', alpha=0.8, color=colors_test,
                              edgecolor='black', linewidth=1.2)
        
        # Add value labels
        for bars in [bars1, bars2]:
            for bar in bars:
                height = bar.get_height()
                axes[idx].text(bar.get_x() + bar.get_width()/2., height,
                             f'{height:.4f}',
                             ha='center', va='bottom', fontsize=8)
        
        axes[idx].set_xlabel('Fold', fontsize=11, fontweight='bold')
        axes[idx].set_ylabel(metric_titles[idx], fontsize=11, fontweight='bold')
        axes[idx].set_title(f'{metric_titles[idx]} Across Folds', 
                           fontsize=12, fontweight='bold')
        axes[idx].set_xticks(x_pos)
        axes[idx].set_xticklabels([f'K{i+1}' for i in range(Kfold_val)])
        axes[idx].legend(fontsize=10)
        axes[idx].grid(True, alpha=0.3, axis='y', linestyle='--')
    
    fig.delaxes(axes[5])
    
    plt.suptitle(f'Cross-Validation Metrics - Parameter Set {set_idx}', 
                 fontsize=16, fontweight='bold', y=1.00)
    plt.tight_layout()
    plt.savefig(f'xgboost_results/metrics_comparison_set{set_idx:02d}.png', 
                dpi=300, bbox_inches='tight')
    plt.show()
    
    print(f"✓ Saved: xgboost_results/metrics_comparison_set{set_idx:02d}.png")

print("\nAll metrics plots generated!")

## Compare All Parameter Sets

In [ ]:
print("=" * 70)
print("COMPARISON OF ALL PARAMETER SETS")
print("=" * 70)

# Create comparison dataframe
comparison_df = pd.DataFrame(all_param_set_results)
comparison_df = comparison_df.round(6)

print("\nTest Performance Comparison:")
display(comparison_df[['param_set', 'test_mae_mean', 'test_rmse_mean', 'test_r2_mean']])

# Save comparison
comparison_df.to_csv('xgboost_results/all_parameter_sets_comparison.csv', index=False)
print("\n✓ Saved: xgboost_results/all_parameter_sets_comparison.csv")

## Comparative Visualization

In [ ]:
# Bar plot comparing test performance across parameter sets
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

metrics_to_compare = [
    ('test_mae_mean', 'Test MAE', 'lower'),
    ('test_rmse_mean', 'Test RMSE', 'lower'),
    ('test_r2_mean', 'Test R²', 'higher')
]

for idx, (metric, title, better) in enumerate(metrics_to_compare):
    x_pos = np.arange(len(comparison_df))
    bars = axes[idx].bar(x_pos, comparison_df[metric], alpha=0.8, 
                        edgecolor='black', linewidth=1.5)
    
    # Color the best performer
    if better == 'lower':
        best_idx = comparison_df[metric].idxmin()
    else:
        best_idx = comparison_df[metric].idxmax()
    bars[best_idx].set_color('gold')
    bars[best_idx].set_edgecolor('darkred')
    bars[best_idx].set_linewidth(2.5)
    
    # Add value labels
    for i, bar in enumerate(bars):
        height = bar.get_height()
        axes[idx].text(bar.get_x() + bar.get_width()/2., height,
                     f'{height:.4f}',
                     ha='center', va='bottom', fontsize=10, fontweight='bold')
    
    axes[idx].set_xlabel('Parameter Set', fontsize=12, fontweight='bold')
    axes[idx].set_ylabel(title, fontsize=12, fontweight='bold')
    axes[idx].set_title(f'{title} Comparison\n(Gold = Best)', 
                       fontsize=13, fontweight='bold')
    axes[idx].set_xticks(x_pos)
    axes[idx].set_xticklabels([f'Set {i+1}' for i in range(len(comparison_df))])
    axes[idx].grid(True, alpha=0.3, axis='y', linestyle='--')

plt.tight_layout()
plt.savefig('xgboost_results/parameter_sets_comparison.png', 
            dpi=300, bbox_inches='tight')
plt.show()

print("✓ Saved: xgboost_results/parameter_sets_comparison.png")

## Final Summary Report 

In [ ]:
print("=" * 70)
print("FINAL SUMMARY")
print("=" * 70)

# Find best parameter set (based on lowest test MAE)
best_set_idx = comparison_df['test_mae_mean'].idxmin() + 1
best_set_params = top_param_sets[best_set_idx - 1]
best_row = comparison_df[comparison_df['param_set'] == best_set_idx].iloc[0]

print(f"\nBest Performing Parameter Set: Set {best_set_idx}")
print("-" * 70)

print("\nParameters:")
for param, value in best_set_params.items():
    print(f"  {param}: {value}")

print(f"\nPerformance Metrics:")
print(f"  Test MAE:  {best_row['test_mae_mean']:.6f} ± {best_row['test_mae_std']:.6f}")
print(f"  Test RMSE: {best_row['test_rmse_mean']:.6f} ± {best_row['test_rmse_std']:.6f}")
print(f"  Test MAPE: {best_row['test_mape_mean']:.6f} ± {best_row['test_mape_std']:.6f}")
print(f"  Test R²:   {best_row['test_r2_mean']:.6f} ± {best_row['test_r2_std']:.6f}")


# Save summary report 
with open('xgboost_results/FINAL_SUMMARY.txt', 'w') as f:
    f.write("=" * 70 + "\n")
    f.write("XGBoost Model Training - Final Summary (CPU-ONLY)\n")
    f.write("=" * 70 + "\n\n")
    
    f.write("Execution Configuration:\n")
    f.write("  Mode: CPU\n")
    f.write("  Tree Method: hist\n")
    f.write("  n_jobs: -1 (All CPU cores)\n")
    f.write(f"  Total Training Time: {search_time/60:.2f} minutes\n")
    
    f.write(f"\nBest Performing Parameter Set: Set {best_set_idx}\n")
    f.write("-" * 70 + "\n")
    f.write("Parameters:\n")
    for param, value in best_set_params.items():
        f.write(f"  {param}: {value}\n")
    
    f.write(f"\nPerformance Metrics:\n")
    f.write(f"  Test MAE:  {best_row['test_mae_mean']:.6f} ± {best_row['test_mae_std']:.6f}\n")
    f.write(f"  Test RMSE: {best_row['test_rmse_mean']:.6f} ± {best_row['test_rmse_std']:.6f}\n")
    f.write(f"  Test MAPE: {best_row['test_mape_mean']:.6f} ± {best_row['test_mape_std']:.6f}\n")
    f.write(f"  Test R²:   {best_row['test_r2_mean']:.6f} ± {best_row['test_r2_std']:.6f}\n")
    
    f.write("\n" + "=" * 70 + "\n")

    f.write("=" * 70 + "\n\n")
    f.write(comparison_df.to_string(index=False))

print("\n✓ Saved: xgboost_results/FINAL_SUMMARY.txt")

print("\n" + "=" * 70)
print("TRAINING COMPLETE! (CPU)")
print("=" * 70)
print(f"\nTotal training time: {search_time/60:.2f} minutes")
print("Acceleration Mode: CPU (Multi-core)")
print("\nGenerated Files:")
print(f"  - {top_n_sets} parity plots in 'xgboost_parity_plots/' folder")
print(f"  - {top_n_sets} detailed metric files in 'xgboost_results/' folder")
print(f"  - Comparison results in 'xgboost_results/' folder")
print(f"  - {top_n_sets * Kfold_val} trained models in 'trained_mod_CBFV_xgboost/' folder")
print("  - Summary report: 'xgboost_results/FINAL_SUMMARY.txt'")
print("=" * 70)


import gc
gc.collect()
print("\n✓ CPU memory cleaned (garbage collection)")

## Per-Composition K-Fold Consistency Analysis

In [ ]:
print("=" * 70)
print("PER-COMPOSITION CONSISTENCY ANALYSIS")
print("=" * 70)

import re
from collections import defaultdict


best_set_idx   = comparison_df['test_mae_mean'].idxmin() + 1
best_param_set = top_param_sets[best_set_idx - 1]
print(f"Using best param set: Set {best_set_idx}")

assert 'alloy_name' in df.columns, (
    "'alloy_name' column not found in df. "
    "Available: " + str(df.columns.tolist())
)


DOPANT_ELEMENTS = ['Pt', 'Pd', 'Ir', 'Rh']

def get_dopant(alloy_name, dopants=DOPANT_ELEMENTS):
    """Return first matching dopant symbol found in alloy_name, else 'NiAl_base'."""
    for d in dopants:
        if re.search(r'(?<![A-Z])' + d + r'(?![a-z])', alloy_name):
            return d
    return 'NiAl_base'


sample_alloys = df['alloy_name'].unique()[:8]
print("\nAlloy name → dopant group (sample check):")
for a in sample_alloys:
    print(f"  {a:35s} → {get_dopant(a)}")


elem_records = defaultdict(list)   # {dopant: [(actual, pred), ...]}
alloy_records = defaultdict(list)  # {alloy_name: [(actual, pred), ...]}

xgb_best = {
    **best_param_set,
    'random_state': random_state,
    'objective'   : 'reg:squarederror',
    'tree_method' : 'hist',
    'n_jobs'      : -1
}

kf_c = KFold(n_splits=Kfold_val, shuffle=True, random_state=random_state)
print(f"\nTracking predictions across {Kfold_val} folds ...")

for fold_i, (tr_idx, te_idx) in enumerate(kf_c.split(X), 1):
    mod_c = xgb.XGBRegressor(**xgb_best)
    mod_c.fit(X[tr_idx], Y[tr_idx],
              eval_set=[(X[tr_idx], Y[tr_idx]), (X[te_idx], Y[te_idx])],
              verbose=False)
    y_pred_te = mod_c.predict(X[te_idx])

    df_test = df.iloc[te_idx]

    # Collect per-dopant and per-alloy predictions
    for i, (actual, pred) in enumerate(zip(Y[te_idx], y_pred_te)):
        alloy_name = df_test.iloc[i]['alloy_name']
        dopant     = get_dopant(alloy_name)
        elem_records[dopant].append((float(actual), float(pred)))
        alloy_records[alloy_name].append((float(actual), float(pred)))

    # Fold-level summary (once per fold)
    mae_f  = mean_absolute_error(Y[te_idx], y_pred_te)
    r2_f   = r2_score(Y[te_idx], y_pred_te)
    n_elem = df_test['alloy_name'].nunique()
    print(f"  Fold {fold_i}: MAE={mae_f:.5f}  R2={r2_f:.4f}  "
          f"({len(te_idx)} rows, {n_elem} compositions)")


def make_stats(label, recs):
    actuals = np.array([r[0] for r in recs])
    preds   = np.array([r[1] for r in recs])
    n_obs   = len(recs)
    mae_c   = float(np.mean(np.abs(actuals - preds)))
    bias_c  = float(np.mean(preds - actuals))
    std_c   = float(np.std(preds)) if n_obs > 1 else 0.0
    r2_c    = float(r2_score(actuals, preds)) if n_obs > 1 else float('nan')
    return {
        'alloy'      : label,
        'n_test_obs' : n_obs,
        'mean_actual': round(float(actuals.mean()), 5),
        'mean_pred'  : round(float(preds.mean()),   5),
        'MAE'        : round(mae_c,  5),
        'bias'       : round(bias_c, 5),
        'pred_std'   : round(std_c,  5),
        'R2'         : round(r2_c,   4) if r2_c == r2_c else float('nan'),
    }


elem_stats = [make_stats(el, recs) for el, recs in elem_records.items()]

if not elem_stats:
    raise RuntimeError(
        "elem_stats is empty — no dopant records were collected. "
        f"Check that alloy_name strings contain one of {DOPANT_ELEMENTS}. "
        f"Sample names: {list(df['alloy_name'].unique()[:5])}"
    )

comp_df = pd.DataFrame(elem_stats).sort_values('MAE').reset_index(drop=True)
comp_df.to_csv('xgboost_results/per_dopant_consistency.csv', index=False)
print(f"\nDopant groups tracked: {len(comp_df)}")
print("Saved -> xgboost_results/per_dopant_consistency.csv")
print("\nPer-dopant-group consistency (sorted by MAE):")
display(comp_df)


alloy_stats = [make_stats(name, recs) for name, recs in alloy_records.items()]
alloy_df    = pd.DataFrame(alloy_stats).sort_values('MAE').reset_index(drop=True)
alloy_df.to_csv('xgboost_results/per_composition_consistency.csv', index=False)
print(f"\nIndividual compositions tracked: {len(alloy_df)}")
print("Saved -> xgboost_results/per_composition_consistency.csv")
print("\n10 best-predicted compositions (lowest MAE):")
display(alloy_df.head(10))
print("\n10 hardest compositions (highest MAE):")
display(alloy_df.tail(10))


In [ ]:
# -- representative subset: best / middle / worst -------------------------
N_SHOW = 24  
n3     = N_SHOW // 3

rep_df = pd.concat([
    comp_df.head(n3),
    comp_df.iloc[len(comp_df)//2 - n3//2 : len(comp_df)//2 + n3//2],
    comp_df.tail(n3)
]).drop_duplicates('alloy').reset_index(drop=True)

q33 = comp_df['MAE'].quantile(0.33)
q67 = comp_df['MAE'].quantile(0.67)

barcol = [
    '#2ecc71' if v <= q33 else '#e74c3c' if v >= q67 else '#3498db'
    for v in rep_df['MAE']
]

xpos = np.arange(len(rep_df))

# Figure 1 - actual vs predicted with error bars --------------------------
fig, ax = plt.subplots(figsize=(16, 6))

ax.scatter(
    xpos,
    rep_df['mean_actual'],
    marker='D',
    s=2,
    c='black',
    zorder=6,
    label='Actual'
)

ax.errorbar(
    xpos,
    rep_df['mean_pred'],
    
    fmt='o',
    ms=7,
    capsize=4,
    elinewidth=1.4,
    color='steelblue',
    ecolor='grey',
    label='Predicted (mean ± std across folds)'
)

ax.set_xticks(xpos)
ax.set_xticklabels(rep_df['alloy'], rotation=55, ha='right', fontsize=8)
ax.set_ylabel('Mass Change', fontsize=12)
ax.set_title(
    f"Per-Composition Consistency – {len(rep_df)} Representative Alloys\n"
    "(green=good | blue=mid | red=challenging)",
    fontsize=12
)

ax.legend(fontsize=10)
ax.grid(axis='y', linestyle='--', alpha=0.4)

plt.tight_layout()
plt.savefig(
    'xgboost_results/comp_actual_vs_pred.png',
    dpi=150,
    bbox_inches='tight'
)
plt.show()
print("Saved -> xgboost_results/comp_actual_vs_pred.png")

# Figure 2 - MAE bar + bias bar -------------------------------------------
fig, axes = plt.subplots(1, 2, figsize=(18, 6))

# MAE Plot
axes[0].bar(
    xpos,
    rep_df['MAE'],
    color=barcol,
    edgecolor='k',
    linewidth=0.5
)
axes[0].set_xticks(xpos)
axes[0].set_xticklabels(rep_df['alloy'], rotation=55, ha='right', fontsize=8)
axes[0].set_ylabel('MAE', fontsize=12)
axes[0].set_title(
    "Per-Composition MAE\n"
    "(green=good | blue=mid | red=hard)",
    fontsize=11
)
axes[0].grid(axis='y', linestyle='--', alpha=0.4)

# Bias Plot
biascol = [
    '#e74c3c' if v > 0 else '#3498db'
    for v in rep_df['bias']
]

axes[1].bar(
    xpos,
    rep_df['bias'],
    color=biascol,
    edgecolor='k',
    linewidth=0.5
)
axes[1].axhline(0, color='k', lw=1.2, ls='--')
axes[1].set_xticks(xpos)
axes[1].set_xticklabels(rep_df['alloy'], rotation=55, ha='right', fontsize=8)
axes[1].set_ylabel('Prediction Bias (pred − actual)', fontsize=12)
axes[1].set_title(
    "Per-Composition Bias\n"
    "(red=over-predicted | blue=under-predicted)",
    fontsize=11
)
axes[1].grid(axis='y', linestyle='--', alpha=0.4)

plt.suptitle(
    'Per-Composition Error & Bias Analysis',
    fontsize=13,
    fontweight='bold'
)
plt.tight_layout()
plt.savefig(
    'xgboost_results/comp_mae_bias.png',
    dpi=150,
    bbox_inches='tight'
)
plt.show()
print("Saved -> xgboost_results/comp_mae_bias.png")

# Figure 3 - consistency map (pred_std vs MAE) ----------------------------
fig, ax = plt.subplots(figsize=(9, 7))

sc = ax.scatter(
    comp_df['pred_std'],
    comp_df['MAE'],
    c=comp_df['mean_actual'],
    cmap='viridis',
    s=55,
    alpha=0.8,
    edgecolors='k',
    linewidths=0.3
)

plt.colorbar(sc, ax=ax, label='Mean Actual Mass Change')

for _, row in rep_df.iterrows():
    ax.annotate(
        row['alloy'],
        (row['pred_std'], row['MAE']),
        fontsize=6,
        alpha=0.85,
        xytext=(3, 2),
        textcoords='offset points'
    )

ax.set_xlabel('Prediction Std Dev (spread across folds)', fontsize=12)
ax.set_ylabel('MAE per Composition', fontsize=12)
ax.set_title(
    "Prediction Consistency Map\n"
    "(bottom-left = consistent & accurate | top-right = inconsistent & inaccurate)",
    fontsize=11
)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(
    'xgboost_results/comp_consistency_map.png',
    dpi=150,
    bbox_inches='tight'
)
plt.show()
print("Saved -> xgboost_results/comp_consistency_map.png")

# Figure 4 - R2 histogram over compositions with >1 test obs ---------------
comp_r2 = comp_df.dropna(subset=['R2'])

if len(comp_r2) > 2:
    mean_r2   = comp_r2['R2'].mean()
    median_r2 = comp_r2['R2'].median()

    fig, ax = plt.subplots(figsize=(9, 5))
    ax.hist(
        comp_r2['R2'],
        bins=20,
        color='steelblue',
        edgecolor='k',
        alpha=0.8
    )

    ax.axvline(
        mean_r2,
        color='red',
        lw=2,
        ls='--',
        label=f"Mean R2 = {mean_r2:.3f}"
    )
    ax.axvline(
        median_r2,
        color='orange',
        lw=2,
        ls='--',
        label=f"Median R2 = {median_r2:.3f}"
    )

    ax.set_xlabel('R2 per Composition', fontsize=12)
    ax.set_ylabel('Count', fontsize=12)
    ax.set_title(
        "R2 Distribution Across Compositions\n"
        "(compositions with >1 test observation)",
        fontsize=12
    )
    ax.legend(fontsize=10)
    ax.grid(True, alpha=0.3)

    plt.tight_layout()
    plt.savefig(
        'xgboost_results/comp_r2_distribution.png',
        dpi=150,
        bbox_inches='tight'
    )
    plt.show()
    print("Saved -> xgboost_results/comp_r2_distribution.png")
else:
    print(
        "Note: each composition appears only once in the test set "
        "- per-composition R2 requires >1 test observation."
    )

print("\n[Cell 14 complete]")

In [ ]:
print("=" * 70)
print("CELL 15 – COMPOSITION-AWARE LEARNING CURVE (15 random states)")
print("=" * 70)

from sklearn.model_selection import train_test_split


TEST_FRAC        = 0.20
COMP_STEP        = 1
N_RANDOM_STATES  = 24                            
RANDOM_SEEDS     = [random_state + i for i in range(N_RANDOM_STATES)]
─

assert 'alloy_name' in df.columns, "'alloy_name' column not found in df!"

print(f"Random seeds used  : {RANDOM_SEEDS}")


xgb_lc_params = {
    **best_param_set,
    'random_state': random_state,
    'objective'   : 'reg:squarederror',
    'tree_method' : 'hist',
    'n_jobs'      : -1
}


all_lc_rows = []   # raw per-seed rows
seed_models = {}   # {seed: model trained on full pool}
seed_meta   = {}   # {seed: (train_pool_idx, test_idx)}

for seed_i, seed in enumerate(RANDOM_SEEDS, 1):
    print(f"\n── Seed {seed_i}/{N_RANDOM_STATES}  (random_state={seed}) {'─'*35}")

  
    df_idx = np.arange(len(df))
    train_pool_idx, test_idx = train_test_split(
        df_idx, test_size=TEST_FRAC, random_state=seed
    )

    X_test_lc = X[test_idx]
    Y_test_lc = Y[test_idx]

    pool_df       = df.iloc[train_pool_idx].copy()
    pool_df['_orig_idx'] = train_pool_idx

    comp_counts   = pool_df['alloy_name'].value_counts()
    all_comps     = comp_counts.index.tolist()
    n_comps_total = len(all_comps)

    steps = list(range(COMP_STEP, n_comps_total, COMP_STEP))
    if steps[-1] != n_comps_total:
        steps.append(n_comps_total)

    print(f"   Train pool: {len(train_pool_idx)} rows | "
          f"Test: {len(test_idx)} rows | "
          f"Compositions: {n_comps_total} | Steps: {len(steps)}")

    for n_comp_step in steps:
        chosen_comps  = all_comps[:n_comp_step]
        mask          = pool_df['alloy_name'].isin(chosen_comps)
        step_orig_idx = pool_df.loc[mask, '_orig_idx'].values

        X_tr = X[step_orig_idx]
        Y_tr = Y[step_orig_idx]

        model_lc = xgb.XGBRegressor(**xgb_lc_params)
        model_lc.fit(X_tr, Y_tr, verbose=False)

        yp_test  = model_lc.predict(X_test_lc)
        yp_train = model_lc.predict(X_tr)

        all_lc_rows.append({
            'seed'           : seed,
            'seed_idx'       : seed_i,
            'n_compositions' : n_comp_step,
            'n_train_rows'   : len(X_tr),
            'r2_test'        : round(r2_score(Y_test_lc, yp_test),       5),
            'mae_test'       : round(mean_absolute_error(Y_test_lc, yp_test), 6),
            'r2_train'       : round(r2_score(Y_tr, yp_train),            5),
            'mae_train'      : round(mean_absolute_error(Y_tr, yp_train), 6),
        })

        if n_comp_step == n_comps_total:
            print(f"   Full model → R2_test={all_lc_rows[-1]['r2_test']:.4f}  "
                  f"MAE_test={all_lc_rows[-1]['mae_test']:.5f}")
            seed_models[seed] = model_lc
            seed_meta[seed]   = (train_pool_idx, test_idx)


lc_all_df = pd.DataFrame(all_lc_rows)
lc_all_df.to_csv('xgboost_results/composition_learning_curve_all_seeds.csv', index=False)
print("\nSaved -> xgboost_results/composition_learning_curve_all_seeds.csv")

# Mean ± std across seeds at each composition step
lc_agg = (
    lc_all_df.groupby('n_compositions')
    .agg(
        n_train_rows_mean = ('n_train_rows', 'mean'),
        r2_test_mean      = ('r2_test',  'mean'),
        r2_test_std       = ('r2_test',  'std'),
        mae_test_mean     = ('mae_test', 'mean'),
        mae_test_std      = ('mae_test', 'std'),
        r2_train_mean     = ('r2_train', 'mean'),
        r2_train_std      = ('r2_train', 'std'),
        mae_train_mean    = ('mae_train','mean'),
        mae_train_std     = ('mae_train','std'),
    )
    .reset_index()
)
lc_agg.to_csv('xgboost_results/composition_learning_curve.csv', index=False)
print("Saved -> xgboost_results/composition_learning_curve.csv  (aggregated)")


median_seed_idx = N_RANDOM_STATES // 2
median_seed     = RANDOM_SEEDS[median_seed_idx]
train_pool_idx, _ = seed_meta[median_seed]

full_lc_model     = seed_models[median_seed]
full_train_X      = X[train_pool_idx]
full_train_Y      = Y[train_pool_idx]
full_train_alloys = df.iloc[train_pool_idx]['alloy_name'].values

print(f"\nDownstream SHAP will use median seed: {median_seed}  "
      f"(train pool: {len(train_pool_idx)} rows)")


lc_df = lc_agg.rename(columns={
    'n_compositions'  : 'n_compositions',
    'n_train_rows_mean': 'n_train_rows',
    'r2_test_mean'    : 'r2_test',
    'mae_test_mean'   : 'mae_test',
    'r2_train_mean'   : 'r2_train',
    'mae_train_mean'  : 'mae_train',
})
print(f"\nFinal step summary (averaged over {N_RANDOM_STATES} seeds):")
print(lc_df.tail(3).to_string(index=False))


In [ ]:
# Reference from full K-fold CV 
ref_row = comparison_df[comparison_df['param_set'] == best_set_idx].iloc[0]
ref_r2  = ref_row['test_r2_mean']
ref_mae = ref_row['test_mae_mean']

nc  = lc_df['n_compositions'].values
ntr = lc_df['n_train_rows'].values.astype(int)

r2_test_mean   = lc_agg['r2_test_mean'].values
r2_test_std    = lc_agg['r2_test_std'].values
mae_test_mean  = lc_agg['mae_test_mean'].values
mae_test_std   = lc_agg['mae_test_std'].values
r2_train_mean  = lc_agg['r2_train_mean'].values
r2_train_std   = lc_agg['r2_train_std'].values
mae_train_mean = lc_agg['mae_train_mean'].values
mae_train_std  = lc_agg['mae_train_std'].values


fig, axes = plt.subplots(1, 2, figsize=(16, 6))

for ax, y_te_m, y_te_s, y_tr_m, y_tr_s, ref, ylabel, title in [
        (axes[0],
         r2_test_mean,  r2_test_std,
         r2_train_mean, r2_train_std,
         ref_r2,  'R²',  'Composition Learning Curve – R²'),
        (axes[1],
         mae_test_mean,  mae_test_std,
         mae_train_mean, mae_train_std,
         ref_mae, 'MAE', 'Composition Learning Curve – MAE')]:

    ax.plot(nc, y_te_m, 'o-', color='tomato',    lw=2.2, ms=6, label='Test (mean)')
    ax.fill_between(nc, y_te_m - y_te_s, y_te_m + y_te_s,
                    color='tomato', alpha=0.20, label='Test ±1 std')

    ax.plot(nc, y_tr_m, 's--', color='steelblue', lw=2.2, ms=6, label='Train (mean)')
    ax.fill_between(nc, y_tr_m - y_tr_s, y_tr_m + y_tr_s,
                    color='steelblue', alpha=0.20, label='Train ±1 std')

    ax.axhline(ref, color='darkgreen', lw=1.6, ls='-.',
               label=f'Full-CV = {ref:.4f}')
    ax.set_xlabel('Number of Compositions in Training Set', fontsize=12)
    ax.set_ylabel(ylabel, fontsize=12)
    ax.set_title(title, fontsize=12)
    ax.set_xticks(nc)
    ax.tick_params(axis='x', rotation=45)
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)

    
    ax2 = ax.twiny()
    ax2.set_xlim(ax.get_xlim())
    ax2.set_xticks(nc)
    ax2.set_xticklabels(ntr, rotation=45, fontsize=7)
    ax2.set_xlabel('Number of Training Rows', fontsize=9, color='grey')

plt.suptitle(
    f'Composition-Aware Learning Curves  |  Best Param Set {best_set_idx}\n'
    f'Mean ± std over {N_RANDOM_STATES} random train/test splits',
    fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('xgboost_results/comp_learning_curve_r2_mae.png',
            dpi=150, bbox_inches='tight')
plt.show()
print("Saved -> xgboost_results/comp_learning_curve_r2_mae.png")


gap_r2  = r2_train_mean  - r2_test_mean
gap_mae = mae_test_mean  - mae_train_mean

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for ax, gap, ylabel, title, col in [
        (axes[0], gap_r2,  'R² Gap (Train − Test)',  'Overfitting Diagnostic – R²',  'darkorchid'),
        (axes[1], gap_mae, 'MAE Gap (Test − Train)', 'Generalisation Gap – MAE',      'chocolate')]:
    ax.plot(nc, gap, 'o-', color=col, lw=2, ms=6)
    ax.fill_between(nc, 0, gap, where=gap > 0,
                    alpha=0.25, color=col, label='Gap region')
    ax.axhline(0, color='k', lw=0.9, ls='--')
    ax.set_xlabel('Number of Compositions', fontsize=12)
    ax.set_ylabel(ylabel, fontsize=12)
    ax.set_title(title, fontsize=12)
    ax.set_xticks(nc)
    ax.tick_params(axis='x', rotation=45)
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.3)

plt.suptitle(f'Train–Test Gap  |  Mean over {N_RANDOM_STATES} seeds',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('xgboost_results/comp_learning_curve_gap.png',
            dpi=150, bbox_inches='tight')
plt.show()
print("Saved -> xgboost_results/comp_learning_curve_gap.png")


d_r2   = np.diff(r2_test_mean)
d_mae  = np.diff(mae_test_mean)
nc_mid = (nc[:-1] + nc[1:]) / 2.0
bw     = (nc[1] - nc[0]) * 0.6 if len(nc) > 1 else 2.0

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for ax, d_vals, ylabel, title, better in [
        (axes[0], d_r2,  'ΔR² per step',  'Marginal R² Gain',       'max'),
        (axes[1], d_mae, 'ΔMAE per step', 'Marginal MAE Reduction',  'min')]:
    colors = ['#27ae60' if (v > 0 if better=='max' else v < 0)
              else '#e74c3c' for v in d_vals]
    ax.bar(nc_mid, d_vals, width=bw, color=colors,
           edgecolor='k', linewidth=0.4)
    ax.axhline(0, color='k', lw=0.9)
    ax.set_xlabel('Midpoint Composition Count', fontsize=11)
    ax.set_ylabel(ylabel, fontsize=11)
    ax.set_title(title, fontsize=11)
    ax.grid(axis='y', linestyle='--', alpha=0.4)

plt.suptitle(f'Marginal Value of Each New Group of Compositions  |  {N_RANDOM_STATES} seeds',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('xgboost_results/comp_learning_curve_marginal.png',
            dpi=150, bbox_inches='tight')
plt.show()
print("Saved -> xgboost_results/comp_learning_curve_marginal.png")


fig, ax = plt.subplots(figsize=(10, 5))
sc = ax.scatter(nc, ntr, c=r2_test_mean, cmap='RdYlGn',
                s=80, edgecolors='k', lw=0.4, zorder=5)
plt.colorbar(sc, ax=ax, label='R² test (mean)')
for x_, y_, r2_ in zip(nc, ntr, r2_test_mean):
    ax.annotate(f'R²={r2_:.3f}', (x_, y_),
                xytext=(4, 4), textcoords='offset points', fontsize=7)
ax.set_xlabel('Number of Compositions in Training', fontsize=12)
ax.set_ylabel('Mean Number of Rows in Training', fontsize=12)
ax.set_title(f'Composition Count vs Row Count — coloured by Mean Test R²  ({N_RANDOM_STATES} seeds)',
             fontsize=12)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('xgboost_results/comp_vs_rows_scatter.png',
            dpi=150, bbox_inches='tight')
plt.show()
print("Saved -> xgboost_results/comp_vs_rows_scatter.png")


last3_dr2  = float(np.mean(np.abs(d_r2[-3:])))  if len(d_r2)  >= 3 else float('nan')
last3_dmae = float(np.mean(np.abs(d_mae[-3:]))) if len(d_mae) >= 3 else float('nan')
print("\n" + "=" * 70)
print("DATA SUFFICIENCY VERDICT")
print("=" * 70)
print(f"  Mean |ΔR²|  over last 3 steps : {last3_dr2:.6f}")
print(f"  Mean |ΔMAE| over last 3 steps : {last3_dmae:.6f}")
